**Desafío Técnico TDM – Generación, validación y trazabilidad de datos sintéticos de clientes**

**1. Introducción**
Este documento define el Desafío Técnico para el rol de Coordinador TDM, combinando una presentación formal del objetivo del área con las especificaciones técnicas del ejercicio.

**Objetivo**
La Unidad de Entornos No Productivos – Chapter QA busca garantizar datos adecuados, reproducibles y trazables para pruebas funcionales y E2E, para esto se solicita generar un dataset sintético de clientes bajo un contrato de datos explícito, validarlo contra reglas de negocio y producir artefactos consumibles con trazabilidad (logs, reportes, perfiles).

**Alcance**: Generación parametrizable de registros, inyección controlada de errores, validación contra contrato YAML, perfilado, logging estructurado y exportación versionada en CSV/JSON.

In [4]:
# =====================================
# 2. SETUP DEL ENTORNO
# Instalación de dependencias
# =====================================
!pip install pyspark pyyaml faker -q

print("✅ Librerías instaladas correctamente")

✅ Librerías instaladas correctamente


In [5]:
# =====================================
# 3. INICIALIZACIÓN DE SPARK
# =====================================
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("TDM_Challenge") \
    .master("local[*]") \
    .getOrCreate()

print("✅ Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

✅ Spark iniciado correctamente
Versión de Spark: 4.0.2


4. Generación de 100–500 registros de clientes sintéticos (Parametrizable)

In [8]:
import random
import math
from datetime import datetime, timedelta

# Parámetros según requerimientos
NUM_REGISTROS = 400
PORC_ERROR = 0.0005  # Representa el 0.05%
SEED = 42

# Reproducibilidad
random.seed(SEED)

# Aseguramos al menos 1 error para trazabilidad de QA
num_errores = max(1, int(NUM_REGISTROS * PORC_ERROR))

# Pre-calculamos los índices que serán "saboteados"
# Usamos un rango de 1 a NUM_REGISTROS para coincidir con tu bucle
indices_con_error = set(random.sample(range(1, NUM_REGISTROS + 1), num_errores))


print("=====================================")
print("⚙️ CONFIGURACIÓN DEL PROCESO")
print("=====================================")
print(f"📊 Total registros: {NUM_REGISTROS}")
print(f"⚠️ Porcentaje error: {PORC_ERROR*100}%")
print(f"❌ Registros con error: {num_errores}")
print(f"🎯 Seed: {SEED}")
print(f"📍 Índices con error: {sorted(indices_con_error)}")

⚙️ CONFIGURACIÓN DEL PROCESO
📊 Total registros: 400
⚠️ Porcentaje error: 0.05%
❌ Registros con error: 1
🎯 Seed: 42
📍 Índices con error: [328]


5. GUARDO PARAMETROS

In [9]:
config = {
    "num_registros": NUM_REGISTROS,
    "porc_error": PORC_ERROR,
    "seed": SEED,
    "num_errores": num_errores
}

# 6. CONTRATO DE DATOS YAML
El contrato fue diseñado con soporte multidentificación para hacer la solución extensible a escenarios futuros, manteniendo coherencia con las reglas del dominio y habilitando validación cruzada según el tipo documental.

In [1]:
# Crear el YAML corregido y mejorado
yaml_content = """
dataset: clientes_no_productivo
version: 1.3
description: Dataset sintético bancario con soporte multidentificación (EC) y validación cruzada
owner: Unidad de Entornos No Productivos

catalogos:
  tipo_identificacion:
    - CEDULA
    - RUC
    - PASAPORTE

  estado_cliente:
    - ACTIVO
    - INACTIVO

tables:
  clientes:
    primary_key: customer_id

    columns:
      customer_id:
        type: string
        required: true
        unique: true
        pattern: "^Cus[0-9]{3}$"
        description: Identificador único autosecuencial del cliente

      tipo_identificacion:
        type: string
        required: true
        catalog: tipo_identificacion
        description: Define la regla de validación para el campo identificacion

      identificacion:
        type: string
        required: true
        unique: true
        sensitive: true
        description: Número de identificación sintético según tipo documental
        rules:
          - if: "tipo_identificacion == 'CEDULA'"
            pattern: "^[0-9]{10}$"
            validator: "modulo_10_ecuador"

          - if: "tipo_identificacion == 'RUC'"
            pattern: "^[0-9]{13}$"
            validator: "modulo_10_o_13_ruc"

          - if: "tipo_identificacion == 'PASAPORTE'"
            pattern: "^[A-Z0-9]{3,15}$"
            description: Alfanumérico estándar internacional

      nombre:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Nombre sintético del cliente

      apellido:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Apellido sintético del cliente

      fecha_nacimiento:
        type: date
        required: true
        min_age: 18
        max_age: 90
        format: "%d-%m-%Y"
        sensitive: true
        description: Fecha de nacimiento válida para clientes entre 18 y 90 años

      email:
        type: string
        required: true
        format: email
        sensitive: true
        description: Correo electrónico sintético del cliente

      telefono:
        type: string
        required: true
        pattern: "^09[0-9]{8}$"
        sensitive: true
        description: Teléfono celular sintético del cliente

      direccion:
        type: string
        required: true
        max_length: 100
        sensitive: true
        description: Dirección física del cliente

      fecha_creacion:
        type: timestamp
        required: true
        format: "%d/%m/%Y %H:%M:%S"
        not_future: true
        description: Fecha y hora de creación del registro

      estado_cliente:
        type: string
        required: true
        catalog: estado_cliente
        description: Estado lógico del cliente

    business_rules:
      - name: edad_minima
        description: La edad del cliente debe ser mayor o igual a 18 años
        rule: "edad >= 18"

      - name: cliente_inactivo_antiguedad
        description: Si el cliente está INACTIVO, la fecha_creacion debe corresponder a una antigüedad mínima de 6 meses
        rule: "if estado_cliente == 'INACTIVO' then antiguedad_meses(fecha_creacion) >= 6"

      - name: email_valido
        description: El correo electrónico debe cumplir formato válido y ser controlado
        rule: "formato_email(email) == true"

      - name: customer_id_unico
        description: El identificador del cliente debe ser único
        rule: "unique(customer_id) == true"

      - name: sin_nulos
        description: Los campos obligatorios no deben contener valores nulos
        rule: "required_fields_not_null == true"
"""

with open("contract.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("✅ YAML corregido y creado correctamente")

✅ YAML corregido y creado correctamente


# 7. CARGA INICIAL DEL CONTRATO YAML                                                            
En esta etapa se carga el contrato de datos definido en YAML, el cual actúa como fuente de verdad para la estructura del dataset, catálogos y reglas asociadas. Además de la lectura del archivo, se realiza una validación mínima de integridad estructural para garantizar que el proceso de generación y validación opere sobre un contrato consistente y trazable.

In [18]:
import yaml

try:
    with open("contract.yaml", "r", encoding="utf-8") as f:
        contract = yaml.safe_load(f)

    # Validación mínima de estructura esperada
    required_keys = ["dataset", "version", "tables", "catalogos"]
    missing_keys = [k for k in required_keys if k not in contract]

    if missing_keys:
        raise ValueError(f"Faltan claves obligatorias en el contrato: {missing_keys}")

    # Variables útiles para los siguientes pasos
    dataset_name = contract["dataset"]
    dataset_version = contract["version"]
    tables = list(contract["tables"].keys())
    columns_clientes = list(contract["tables"]["clientes"]["columns"].keys())
    catalogs = list(contract["catalogos"].keys())

    print("✅ Contrato YAML leído y validado correctamente")
    print("=====================================")
    print(f"📦 Dataset      : {dataset_name}")
    print(f"🏷️ Versión      : {dataset_version}")
    print(f"🗂️ Tablas       : {tables}")
    print(f"🧱 Columnas     : {columns_clientes}")
    print(f"📚 Catálogos    : {catalogs}")

except FileNotFoundError:
    print("❌ No se encontró el archivo contract.yaml")
except yaml.YAMLError as e:
    print("❌ Error de sintaxis en el archivo YAML")
    print(e)
except Exception as e:
    print("❌ Error al cargar el contrato")
    print(e)

✅ Contrato YAML leído y validado correctamente
📦 Dataset      : clientes_no_productivo
🏷️ Versión      : 1.3
🗂️ Tablas       : ['clientes']
🧱 Columnas     : ['customer_id', 'tipo_identificacion', 'identificacion', 'nombre', 'apellido', 'fecha_nacimiento', 'email', 'telefono', 'direccion', 'fecha_creacion', 'estado_cliente']
📚 Catálogos    : ['tipo_identificacion', 'estado_cliente']


# 8. INSPECCIÓN Y VALIDACIÓN DEL CONTRATO DE DATOS
En esta etapa se carga el contrato de datos definido en YAML y se realiza una validación inicial de su estructura. El objetivo es asegurar que el proceso de generación y validación del dataset opere sobre una fuente de verdad consistente, trazable y reutilizable. Además, se inspeccionan las claves principales, columnas, catálogos y reglas de negocio para preparar los siguientes pasos del flujo TDM.

In [21]:
import yaml

try:
    # 1. Cargar el contrato YAML
    with open("contract.yaml", "r", encoding="utf-8") as f:
        contract = yaml.safe_load(f)

    # 2. Validación mínima de estructura esperada
    required_keys = ["dataset", "version", "tables", "catalogos"]
    missing_keys = [k for k in required_keys if k not in contract]

    if missing_keys:
        raise ValueError(f"Faltan claves obligatorias en el contrato: {missing_keys}")

    # 3. Variables útiles para los siguientes pasos
    dataset_name = contract["dataset"]
    dataset_version = contract["version"]
    tables = list(contract["tables"].keys())

    tabla_clientes = contract["tables"]["clientes"]
    columnas_clientes = tabla_clientes["columns"]
    reglas_negocio = tabla_clientes.get("business_rules", [])

    catalogo_tipo_id = contract["catalogos"]["tipo_identificacion"]
    catalogo_estado = contract["catalogos"]["estado_cliente"]

    # 4. Resumen general del contrato
    print("✅ Contrato YAML leído y validado correctamente")
    print("=====================================")
    print(f"📦 Dataset        : {dataset_name}")
    print(f"🏷️ Versión        : {dataset_version}")
    print(f"🗂️ Tablas         : {tables}")
    print(f"🧱 Columnas       : {list(columnas_clientes.keys())}")
    print(f"📚 Catálogos      : {list(contract['catalogos'].keys())}")

    # 5. Inspección de reglas clave
    print("\n🔍 INSPECCIÓN DE REGLAS DEL CONTRATO")
    print("=====================================")
    print("Primary key:", tabla_clientes["primary_key"])

    print("\nRegla customer_id:")
    print(columnas_clientes["customer_id"])

    print("\nRegla tipo_identificacion:")
    print(columnas_clientes["tipo_identificacion"])

    print("\nRegla identificacion:")
    print(columnas_clientes["identificacion"])

    print("\nCatálogo tipo_identificacion:")
    print(catalogo_tipo_id)

    print("\nCatálogo estado_cliente:")
    print(catalogo_estado)

    print("\nBusiness rules:")
    if reglas_negocio:
        for regla in reglas_negocio:
            print(regla)
    else:
        print("No se definieron reglas de negocio en el contrato.")

except FileNotFoundError:
    print("❌ No se encontró el archivo contract.yaml")

except yaml.YAMLError as e:
    print("❌ Error de sintaxis en el archivo YAML")
    print(e)

except Exception as e:
    print("❌ Error al cargar o validar el contrato")
    print(e)

✅ Contrato YAML leído y validado correctamente
📦 Dataset        : clientes_no_productivo
🏷️ Versión        : 1.3
🗂️ Tablas         : ['clientes']
🧱 Columnas       : ['customer_id', 'tipo_identificacion', 'identificacion', 'nombre', 'apellido', 'fecha_nacimiento', 'email', 'telefono', 'direccion', 'fecha_creacion', 'estado_cliente']
📚 Catálogos      : ['tipo_identificacion', 'estado_cliente']

🔍 INSPECCIÓN DE REGLAS DEL CONTRATO
Primary key: customer_id

Regla customer_id:
{'type': 'string', 'required': True, 'unique': True, 'pattern': '^Cus[0-9]{3}$', 'description': 'Identificador único autosecuencial del cliente'}

Regla tipo_identificacion:
{'type': 'string', 'required': True, 'catalog': 'tipo_identificacion', 'description': 'Define la regla de validación para el campo identificacion'}

Regla identificacion:
{'type': 'string', 'required': True, 'unique': True, 'sensitive': True, 'description': 'Número de identificación sintético según tipo documental', 'rules': [{'if': "tipo_identifi

In [20]:
from google.colab import files
files.download("contract.yaml")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. GENERACIÓN DE DATOS SINTÉTICOS
En esta etapa se definen las funciones base para la generación del dataset sintético de clientes. La lógica toma como referencia los catálogos y restricciones declaradas en el contrato YAML, con el fin de producir registros consistentes, reproducibles y alineados con el dominio del problema. Se incorporan generadores específicos para identificación, fechas, datos de contacto y dirección, priorizando realismo sintético y trazabilidad.

## 9.1 Configuración de catálogos y dependencias

In [23]:
import random
from faker import Faker
from datetime import datetime, timedelta

# Faker en español
fake = Faker("es_ES")

# Reproducibilidad (MUY IMPORTANTE)
fake.seed_instance(SEED)

# Catálogos desde el contrato YAML (fuente de verdad)
tipos_id = catalogo_tipo_id
estados = catalogo_estado

# Diccionarios sintétics base
nombres = ["Juan", "María", "Carlos", "Ana", "Luis", "Sofía", "Pedro", "Lucía"]
apellidos = ["Pérez", "Gómez", "Torres", "López", "Martínez", "Ramírez", "Vera", "Mora"]

print("✅ Configuración de generación lista")
print("Tipos ID:", tipos_id)
print("Estados:", estados)

✅ Configuración de generación lista
Tipos ID: ['CEDULA', 'RUC', 'PASAPORTE']
Estados: ['ACTIVO', 'INACTIVO']


# 9.2 FUNCIONES DE GENERACIÓN DE IDENTIFICACIONES
En esta subetapa se implementan funciones específicas para la generación de identificaciones sintéticas válidas según el tipo documental. Para cédula se aplica el algoritmo ecuatoriano de módulo 10, para RUC natural se construye a partir de una cédula válida más el sufijo 001, y para pasaporte se utiliza un patrón alfanumérico sintético. Esta lógica asegura coherencia de dominio y realismo en los datos generados.

9.2.1 Calcular dígito verificador

In [29]:
def calcular_digito_verificador_cedula(base9: str) -> int:
    """
    Calcula el dígito verificador de una cédula ecuatoriana
    a partir de los primeros 9 dígitos usando módulo 10.
    """
    if not base9.isdigit() or len(base9) != 9:
        raise ValueError("base9 debe contener exactamente 9 dígitos numéricos.")

    coeficientes = [2, 1, 2, 1, 2, 1, 2, 1, 2]
    suma = 0

    for i, digito in enumerate(base9):
        valor = int(digito) * coeficientes[i]
        if valor >= 10:
            valor -= 9
        suma += valor

    digito_verificador = (10 - (suma % 10)) % 10
    return digito_verificador

9.2.2 Generar cédula ecuatoriana válida

In [30]:
def generar_cedula_ec(provincia: str = "17") -> str:
    """
    Genera una cédula ecuatoriana sintética válida.
    Por defecto usa provincia 17 (Pichincha / Quito).
    """
    provincias_validas = [f"{i:02d}" for i in range(1, 25)]
    if provincia not in provincias_validas:
        raise ValueError("La provincia debe estar entre 01 y 24.")

    tercer_digito = str(random.randint(0, 5))
    siguientes_seis = "".join(str(random.randint(0, 9)) for _ in range(6))

    base9 = provincia + tercer_digito + siguientes_seis
    verificador = calcular_digito_verificador_cedula(base9)

    return base9 + str(verificador)

9.2.3 Validar cédula

In [31]:
def validar_cedula_ec(cedula: str) -> bool:
    """
    Valida una cédula ecuatoriana de 10 dígitos.
    """
    if not cedula.isdigit() or len(cedula) != 10:
        return False

    provincia = int(cedula[:2])
    tercer_digito = int(cedula[2])

    if provincia < 1 or provincia > 24:
        return False

    if tercer_digito < 0 or tercer_digito > 5:
        return False

    base9 = cedula[:9]
    verificador_real = int(cedula[9])
    verificador_calculado = calcular_digito_verificador_cedula(base9)

    return verificador_real == verificador_calculado

9.2.4 Generar RUC natural

In [32]:
def generar_ruc_natural_ec(provincia: str = "17") -> str:
    """
    Genera un RUC sintético de persona natural:
    cédula válida + 001
    """
    return generar_cedula_ec(provincia) + "001"

9.2.5 Generar pasaporte sintético

In [33]:
def generar_pasaporte() -> str:
    """
    Genera un pasaporte sintético alfanumérico.
    """
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=8))

9.2.6. Generar identificación según tipo

In [34]:
def generar_identificacion(tipo: str) -> str:
    if tipo == "CEDULA":
        return generar_cedula_ec("17")
    elif tipo == "RUC":
        return generar_ruc_natural_ec("17")
    elif tipo == "PASAPORTE":
        return generar_pasaporte()
    else:
        raise ValueError(f"Tipo de identificación no soportado: {tipo}")

9.2.7 Validargeneracion de identificaciones

In [36]:
print("Cédula:", generar_cedula_ec())
print("RUC:", generar_ruc_natural_ec())
print("Pasaporte:", generar_pasaporte())

cedula_prueba = generar_cedula_ec()
print("Cédula de prueba:", cedula_prueba)
print("¿Cédula válida?:", validar_cedula_ec(cedula_prueba))

Cédula: 1701615591
RUC: 1720781614001
Pasaporte: T3W5UZBI
Cédula de prueba: 1721316477
¿Cédula válida?: True


# 9.3 GENERAR FECHA DE NACIMIENTO (18–90) Se aplica Regla 1: Edad ≥ 18

In [39]:
def generar_fecha_nacimiento(min_edad: int = 18, max_edad: int = 90) -> str:
    """
    Genera una fecha de nacimiento sintética coherente.

    Reglas:
    - Edad mínima: 18 años
    - Edad máxima: 90 años
    - Se usa una fecha base fija para reproducibilidad
    """

    # Fecha base controlada
    fecha_base = datetime(2025, 1, 1)

    # Generar edad dentro del rango permitido
    edad = random.randint(min_edad, max_edad)

    # Convertir edad a días (considerando bisiestos)
    dias_edad = int(edad * 365.25)

    # Calcular fecha de nacimiento
    fecha = fecha_base - timedelta(days=dias_edad)

    # Variación adicional de días para mayor realismo
    fecha -= timedelta(days=random.randint(0, 364))

    return fecha.strftime("%d/%m/%Y")

# 9.3.1 Funcion de validación

In [40]:
def calcular_edad(fecha_nacimiento_str: str) -> int:
    """
    Calcula la edad a partir de la fecha de nacimiento.
    """
    fecha_nacimiento = datetime.strptime(fecha_nacimiento_str, "%d/%m/%Y")
    fecha_base = datetime(2025, 1, 1)

    edad = fecha_base.year - fecha_nacimiento.year
    if (fecha_base.month, fecha_base.day) < (fecha_nacimiento.month, fecha_nacimiento.day):
        edad -= 1

    return edad

# 9.3.2 Validación y pruebas

In [41]:
fecha = generar_fecha_nacimiento()
edad = calcular_edad(fecha)

print("Fecha:", fecha)
print("Edad:", edad)
print("¿Cumple regla (>=18)?", edad >= 18)

Fecha: 10/10/1960
Edad: 64
¿Cumple regla (>=18)? True


# 9.4 GENERAR EMAIL
Regla 3 (Email Coherente y Controlado)
En esta subetapa se implementa la generación de correos electrónicos coherentes con el nombre y apellido del cliente. Para asegurar consistencia, se normalizan los textos removiendo tildes, espacios y caracteres especiales. Además, se incorpora una función específica para producir correos inválidos de forma controlada, con el fin de probar la Regla 3 del desafío, relacionada con formato y control del email.

In [42]:
import unicodedata
import re

DOMINIO_EMAIL = "testmail.com"

def limpiar_texto_email(texto: str) -> str:
    """
    Normaliza un texto para uso en correos electrónicos:
    - convierte a minúsculas
    - elimina tildes
    - elimina espacios
    - elimina caracteres no alfanuméricos
    """
    texto = texto.lower().strip()

    # Quitar tildes
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    # Eliminar caracteres no permitidos para simplificar el correo
    texto = re.sub(r"[^a-z0-9]", "", texto)

    return texto


def generar_email_invalido(nombre: str, apellido: str, indice: int) -> str:
    """
    Genera formatos inválidos de email para pruebas de QA.
    """
    n_limpio = limpiar_texto_email(nombre)
    a_limpio = limpiar_texto_email(apellido)

    opciones_error = [
        f"{n_limpio}.{a_limpio}{indice}_sin_arroba.com",  # Falta @
        f"{n_limpio}.{a_limpio}{indice}@",               # Falta dominio
        f"error_formato_{indice}",                       # Texto no válido
        f"{n_limpio}{a_limpio}{indice}@dominio",         # Dominio incompleto
    ]

    return random.choice(opciones_error)


def generar_email_coherente(nombre: str, apellido: str, indice: int, con_error: bool = False) -> str:
    """
    Genera un correo electrónico sintético coherente con nombre y apellido.

    Regla de negocio:
    - Debe tener formato válido
    - Debe ser coherente con el nombre del cliente

    Si con_error=True, genera un email inválido para pruebas de calidad.
    """
    n_limpio = limpiar_texto_email(nombre)
    a_limpio = limpiar_texto_email(apellido)

    if con_error:
        return generar_email_invalido(nombre, apellido, indice)

    return f"{n_limpio}.{a_limpio}{indice}@{DOMINIO_EMAIL}"

# 9.4.1 VALIDACIÓN GENERACION MAIL SINTETICO

In [43]:
print(generar_email_coherente("Lucía", "Gómez", 1))
print(generar_email_coherente("María José", "Pérez-López", 2))
print(generar_email_coherente("Carlos", "Torres", 3, con_error=True))

lucia.gomez1@testmail.com
mariajose.perezlopez2@testmail.com
error_formato_3


# 9.5 GENERAR DIRECCION
En esta subetapa se implementa la generación de direcciones sintéticas con formato estructurado basado en convenciones urbanas del Ecuador. La dirección se compone de calle, numeración, referencia y ciudad. Adicionalmente, se define una variante de generación inválida para permitir la validación de calidad de datos, asegurando el cumplimiento de reglas de formato, completitud y longitud máxima.

In [57]:
# Catálogos geográficos sintéticos (Ecuador)
calles_ec = [
    "Av. Amazonas", "Av. 6 de Diciembre", "Av. Colón",
    "Av. Naciones Unidas", "Av. República", "Av. Shyris",
    "Calle Bolívar", "Calle Sucre", "Av. 9 de Octubre"
]

referencias_ec = [
    "y Naciones Unidas", "y Colón", "y Patria",
    "y 10 de Agosto", "y Orellana", "y Mariana de Jesús",
    "y Loja", "y Sucre"
]

ciudades_ec = ["Quito", "Guayaquil", "Cuenca", "Manta", "Ambato"]


def generar_direccion_valida() -> str:
    """
    Genera una dirección sintética válida estilo Ecuador.

    Formato:
    <calle> N<número> <referencia>, <ciudad>

    Ejemplo:
    Av. Amazonas N1234 y Colón, Quito
    """
    calle = random.choice(calles_ec)
    referencia = random.choice(referencias_ec)
    numero = random.randint(100, 9999)
    ciudad = random.choice(ciudades_ec)

    direccion = f"{calle} N{numero} {referencia}, {ciudad}"

    # Regla: longitud máxima 100 caracteres
    return direccion[:100]


def generar_direccion_invalida() -> str:
    """
    Genera direcciones inválidas para pruebas de calidad (QA).
    """
    opciones_error = [
        "SIN_DIRECCION",
        "12345",
        "direccion_invalida@@@",
        "",  # vacío
        "Quito",  # incompleta
    ]
    return random.choice(opciones_error)


def generar_direccion(con_error: bool = False) -> str:
    """
    Genera una dirección sintética.

    - Si con_error=True, genera un valor inválido
    - Caso contrario, genera una dirección válida
    """
    if con_error:
        return generar_direccion_invalida()

    return generar_direccion_valida()


# 9.5.1 Validación direcciones sintéticas

In [58]:
print(generar_direccion())
print(generar_direccion())
print(generar_direccion(con_error=True))

Av. Naciones Unidas N2173 y Naciones Unidas, Quito
Av. Colón N3505 y Naciones Unidas, Quito
direccion_invalida@@@


## 9.6: Regla 2: Coherencia de Clientes Inactivos (≥ 6 meses) Para cumplir con el 0.05% de error.
En esta subetapa se implementa la lógica temporal asociada al estado del cliente. Se aplica la Regla 2 del desafío, según la cual un cliente inactivo debe reflejar una antigüedad mínima de seis meses en su fecha de creación. Adicionalmente, se incorpora una estrategia de inyección de error que genera clientes inactivos con fechas recientes, permitiendo validar controles de calidad y consistencia temporal.

In [59]:
def generar_fechas_y_estado(con_error: bool = False):
    """
    Genera el estado del cliente y su fecha de creación de forma coherente
    con la Regla 2 del desafío:

    Regla 2:
    - Si el cliente está INACTIVO, su fecha_creacion debe reflejar
      una antigüedad mínima de 6 meses (180 días).

    Comportamiento:
    - Caso normal:
        * ACTIVO   -> fecha_creacion reciente (0 a 179 días)
        * INACTIVO -> fecha_creacion antigua (180 a 730 días)
    - Caso con error:
        * fuerza un cliente INACTIVO con fecha_creacion demasiado reciente,
          rompiendo deliberadamente la regla para validación QA.

    Retorna:
    - estado_cliente (str)
    - fecha_creacion formateada como string "%d/%m/%Y %H:%M:%S"
    """

    # Fecha base fija para garantizar reproducibilidad
    fecha_base = datetime(2025, 1, 1, 12, 0, 0)

    # Catálogo de estados desde configuración ya cargada
    estado = random.choice(estados)

    if con_error:
        # SABOTAJE QA:
        # Cliente INACTIVO con antigüedad inválida (< 6 meses)
        estado = "INACTIVO"
        fecha_creacion = fecha_base - timedelta(days=1)

    else:
        if estado == "INACTIVO":
            # Cumple regla: al menos 180 días de antigüedad
            dias_antiguedad = random.randint(180, 730)  # 6 meses a 2 años
            fecha_creacion = fecha_base - timedelta(days=dias_antiguedad)
        else:
            # ACTIVO: creación reciente
            dias_antiguedad = random.randint(0, 179)
            fecha_creacion = fecha_base - timedelta(days=dias_antiguedad)

    return estado, fecha_creacion.strftime("%d/%m/%Y %H:%M:%S")

# 9.6.1 Validación Tipo Clientes Clientes

In [60]:
print("Caso normal:")
print(generar_fechas_y_estado(False))

print("\nCaso con error:")
print(generar_fechas_y_estado(True))

Caso normal:
('ACTIVO', '06/11/2024 12:00:00')

Caso con error:
('INACTIVO', '31/12/2024 12:00:00')


# 9.7 Regla 4: customer_id Único (y su Sabotaje)
En esta subetapa se implementa la generación del identificador único del cliente (customer_id) siguiendo un formato autosecuencial (Cus001, Cus002, …). Este identificador cumple la Regla 4 del contrato, que establece la unicidad del cliente. Adicionalmente, se incorpora una función de sabotaje que reutiliza deliberadamente un identificador existente para simular errores de duplicidad y validar los controles de calidad.

In [63]:
LONGITUD_ID = 3


def generar_customer_id(indice: int) -> str:

    """ Genera un customer_id único con formato:
    Cus001, Cus002, Cus003, ...
    Regla aplicada: - customer_id debe ser único  """

    if not isinstance(indice, int) or indice <= 0:
        raise ValueError("El índice debe ser un entero positivo.")

    return f"Cus{str(indice).zfill(LONGITUD_ID)}"


def generar_customer_id_duplicado() -> str:
    """
    Genera intencionalmente un customer_id duplicado
    para pruebas de calidad y detección de duplicidad.
    """

    # Reutilizamos el primer ID para forzar duplicado
    return generar_customer_id(1)

# Validación Customer id

In [64]:
print(generar_customer_id(1))
print(generar_customer_id(25))
print(generar_customer_id(123))

print("Duplicado:", generar_customer_id_duplicado())

Cus001
Cus025
Cus123
Duplicado: Cus001


# 9.8 GENERAR TELEFONO

In [78]:
def generar_telefono() -> str:
    """
    Genera un teléfono celular sintético ecuatoriano.
    """
    return "09" + "".join(str(random.randint(0, 9)) for _ in range(8))

# 9.9. INYECCIÓN DE FALLAS Script de Inyección de Fallas (Lógica Centralizada)
En esta subetapa se implementa la inyección determinística de errores para validar la robustez del proceso de calidad. Las fallas se clasifican en cuatro categorías típicas de TDM: esquema, dominio, duplicidad y negocio. Cada error se identifica además con un código específico, permitiendo trazabilidad, auditoría y análisis posterior en los reportes de validación.



In [76]:
def inyectar_falla_especifica(cliente: dict, indice: int):
    """
    Inyecta una falla determinística sobre un registro sintético
    según cuatro categorías de error TDM:

    - schema   : rompe tipo de dato o nulidad
    - domain   : rompe catálogo o catálogo permitido
    - dup      : rompe unicidad
    - business : rompe regla de negocio temporal

    Retorna:
    - cliente modificado
    - categoria_error
    - codigo_error
    - descripcion_error
    """

    tipos_error = ["schema", "domain", "dup", "business"]
    categoria_error = tipos_error[indice % len(tipos_error)]

    if categoria_error == "schema":
        # Regla 5: campos no deben contener valores nulos / tipo válido
        cliente["identificacion"] = None
        cliente["nombre"] = 12345

        codigo_error = "SCH001"
        descripcion_error = "Identificación nula y nombre con tipo inválido"

    elif categoria_error == "domain":
          # Rompe dominio del dato:
        # - email inválido
        # - valores fuera del catálogo permitido
        cliente["email"] = generar_email_coherente(
            cliente["nombre"],
            cliente["apellido"],
            indice,
            True
        )
        cliente["tipo_identificacion"] = "CEDULA_FALSA"
        cliente["estado_cliente"] = "DESCONOCIDO"

        codigo_error = "DOM001"
        descripcion_error = "Email inválido y valores fuera del catálogo en tipo_identificacion y estado_cliente"

    elif categoria_error == "dup":
        # Regla 4: customer_id debe ser único
        cliente["customer_id"] = generar_customer_id_duplicado()

        codigo_error = "DUP001"
        descripcion_error = "customer_id duplicado intencionalmente"

    elif categoria_error == "business":
        # Regla 2: cliente INACTIVO debe tener fecha_creacion >= 6 meses
        estado, fecha_creacion = generar_fechas_y_estado(con_error=True)
        cliente["estado_cliente"] = estado
        cliente["fecha_creacion"] = fecha_creacion

        codigo_error = "BUS001"
        descripcion_error = "Cliente INACTIVO con fecha_creacion reciente"

    return cliente, categoria_error, codigo_error, descripcion_error

# 9.10 GENERAR REGISTRO CLIENTE
En esta subetapa se construye cada registro sintético de cliente integrando los diferentes generadores definidos previamente: identificaciones, nombres, fechas, email, teléfono y dirección. El registro se genera inicialmente de forma válida y coherente con el contrato YAML. Posteriormente, si el índice del registro ha sido marcado para error, se aplica una falla controlada y se registra en una estructura de auditoría para asegurar trazabilidad durante el proceso de validación.


In [79]:
# Lista para auditoría de errores inyectados
log_errores_inyectados = []

def generar_cliente(i: int) -> dict:
    """
    Genera un registro sintético de cliente alineado al contrato YAML.

    Lógica:
    - Construye un registro válido por defecto
    - Si el índice está marcado en indices_con_error, aplica una falla controlada
    - Registra la traza del error inyectado para auditoría y reportes
    """

    tiene_error = i in indices_con_error

    # 1. Generación base coherente
    nombre = random.choice(nombres)
    apellido = random.choice(apellidos)
    tipo_id = random.choice(tipos_id)
    estado, fecha_creacion = generar_fechas_y_estado(False)

    cliente = {
        "customer_id": generar_customer_id(i),
        "tipo_identificacion": tipo_id,
        "identificacion": generar_identificacion(tipo_id),
        "nombre": nombre,
        "apellido": apellido,
        "fecha_nacimiento": generar_fecha_nacimiento(),
        "email": generar_email_coherente(nombre, apellido, i, False),
        "telefono": generar_telefono(),
        "direccion": generar_direccion(),
        "fecha_creacion": fecha_creacion,
        "estado_cliente": estado
    }

    # 2. Inyección controlada de fallas
    if tiene_error:
        cliente, categoria_error, codigo_error, descripcion_error = inyectar_falla_especifica(cliente, i)

        log_errores_inyectados.append({
            "indice": i,
            "customer_id": cliente["customer_id"],
            "tipo_identificacion": cliente["tipo_identificacion"],
            "categoria_error": categoria_error,
            "codigo_error": codigo_error,
            "descripcion_error": descripcion_error,
            "timestamp": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        })

    return cliente

# 9.11 GENERAR DATASET Y PASAR A SPARK
En esta etapa se materializa el dataset sintético a partir de la función generar_cliente(), que integra la lógica de identificación, datos personales, contacto, dirección, estado y fecha de creación. Posteriormente, la lista de registros se convierte en un DataFrame de Spark mediante un esquema explícito, lo que permite estandarizar la estructura para los procesos de validación, perfilado y exportación.

In [89]:
# --- CELDA DE IMPORTACIÓN ---
from pyspark.sql.types import StructType, StructField, StringType


# --- CELDA DE DEFINICIÓN DEL ESQUEMA ---
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("tipo_identificacion", StringType(), True),
    StructField("identificacion", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("apellido", StringType(), True),
    StructField("fecha_nacimiento", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("direccion", StringType(), True),
    StructField("fecha_creacion", StringType(), True),
    StructField("estado_cliente", StringType(), True)
])


# 1. GeneraR la lista de diccionarios
data = [generar_cliente(i) for i in range(1, NUM_REGISTROS + 1)]

# 2. CreaR el DataFrame pasando el SCHEMA como segundo argumento
df = spark.createDataFrame(data, schema=schema)


# 3. Resumen de generación
print("✅ DataFrame Spark creado correctamente")
print(f"📊 Total registros generados: {df.count()}")
print(f"🚩 Total errores inyectados: {len(log_errores_inyectados)}")

# 4. Vista previa
df.show(10, truncate=False)

✅ DataFrame Spark creado correctamente
📊 Total registros generados: 400
🚩 Total errores inyectados: 4
+-----------+-------------------+--------------+------+--------+----------------+---------------------------+----------+--------------------------------------------------+-------------------+--------------+
|customer_id|tipo_identificacion|identificacion|nombre|apellido|fecha_nacimiento|email                      |telefono  |direccion                                         |fecha_creacion     |estado_cliente|
+-----------+-------------------+--------------+------+--------+----------------+---------------------------+----------+--------------------------------------------------+-------------------+--------------+
|Cus001     |PASAPORTE          |5O1CE7B5      |Pedro |Mora    |17/10/1995      |pedro.mora1@testmail.com   |0993632280|Av. Colón N9945 y Sucre, Ambato                   |07/04/2024 12:00:00|INACTIVO      |
|Cus002     |CEDULA             |1710967389    |Juan  |Gómez   |17/06/

# 10 VALIDACIÓN DETERMINISTICA DE DATOS GENERADOS

In [91]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def validar_y_reportar(df):
    print("🔎 Iniciando Validación Determinística de Datos...")

    total_registros = df.count()
    # Usamos la fecha actual para las reglas de negocio (2026)
    hoy = F.current_date()

   # 1. Clasificación de Errores
    # ---------------------------------------------------------
    # SCHEMA:
    # - Identificacion es nula
    # - O el nombre contiene solo dígitos (usamos RegEx para evitar el error de CAST)
    df_check = df.withColumn("err_schema",
        F.when(
            F.col("identificacion").isNull() |
            F.col("nombre").rlike("^[0-9]+$"),
            1
        ).otherwise(0))

    # DOMAIN: Valores fuera de ACTIVO/INACTIVO o tipos ID inválidos
    df_check = df_check.withColumn("err_domain",
        F.when(~F.col("estado_cliente").isin("ACTIVO", "INACTIVO") |
               ~F.col("tipo_identificacion").isin("CEDULA", "RUC", "PASAPORTE"), 1).otherwise(0))

    # DUP: Unicidad de customer_id
    w = Window.partitionBy("customer_id")
    df_check = df_check.withColumn("err_dup",
        F.when(F.count("customer_id").over(w) > 1, 1).otherwise(0))

    # BUSINESS: Regla 1 (Edad < 18) y Regla 2 (Inactivo < 6 meses)
    df_check = df_check.withColumn("fnac", F.to_date(F.col("fecha_nacimiento"), "dd/MM/yyyy"))
    df_check = df_check.withColumn("fcrea", F.to_date(F.col("fecha_creacion"), "dd/MM/yyyy HH:mm:ss"))

    df_check = df_check.withColumn("err_business", F.when(
        (F.floor(F.months_between(hoy, F.col("fnac"))/12) < 18) |
        ((F.col("estado_cliente") == "INACTIVO") & (F.months_between(hoy, F.col("fcrea")) < 6)),
        1).otherwise(0))

    # 2. Cálculo de Métricas
    # ---------------------------------------------------------
    resumen = df_check.select(
        F.sum("err_schema").alias("S"), F.sum("err_domain").alias("D"),
        F.sum("err_dup").alias("U"), F.sum("err_business").alias("B")
    ).collect()[0]

    err_totales = (resumen['S'] or 0) + (resumen['D'] or 0) + (resumen['U'] or 0) + (resumen['B'] or 0)
    cumplimiento = ((total_registros - err_totales) / total_registros) * 100

    # 3. Reporte Final
    # ---------------------------------------------------------
    print("\n" + "="*45)
    print("      REPORTE DE MÉTRICAS DE CALIDAD TDM      ")
    print("="*45)
    print(f"Total Registros:      {total_registros}")
    print(f"Reglas Evaluadas:     5 (R1, R2, R3, R4, R5)")
    print(f"Errores Totales:      {int(err_totales)}")
    print(f"% Cumplimiento:       {cumplimiento:.2f}%")
    print("-" * 45)
    print(f"Clasificación de Errores:")
    print(f"  - Schema (Nulos/Tipos): {int(resumen['S'] or 0)}")
    print(f"  - Domain (Catálogos):   {int(resumen['D'] or 0)}")
    print(f"  - Dup (Unicidad):       {int(resumen['U'] or 0)}")
    print(f"  - Business (Lógica):    {int(resumen['B'] or 0)}")
    print("-" * 45)
    print("Muestra de Errores Detectados:")
    df_check.filter("err_schema=1 OR err_domain=1 OR err_dup=1 OR err_business=1") \
            .select("customer_id", "estado_cliente", "fecha_nacimiento").show(5)

# EJECUCIÓN FINAL
validar_y_reportar(df)

🔎 Iniciando Validación Determinística de Datos...

      REPORTE DE MÉTRICAS DE CALIDAD TDM      
Total Registros:      400
Reglas Evaluadas:     5 (R1, R2, R3, R4, R5)
Errores Totales:      1
% Cumplimiento:       99.75%
---------------------------------------------
Clasificación de Errores:
  - Schema (Nulos/Tipos): 1
  - Domain (Catálogos):   0
  - Dup (Unicidad):       0
  - Business (Lógica):    0
---------------------------------------------
Muestra de Errores Detectados:
+-----------+--------------+----------------+
|customer_id|estado_cliente|fecha_nacimiento|
+-----------+--------------+----------------+
|     Cus328|      INACTIVO|      13/11/1953|
+-----------+--------------+----------------+



# 11 FORMATOS DE SALIDA


In [108]:
SEED = 42
random.seed(SEED)
fake.seed_instance(SEED)

In [109]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
nombre_base = f"clientes_{timestamp}_seed{SEED}"

print(nombre_base)

clientes_20260405_1906_seed42


11.1 Exportar CSV

In [110]:
ruta_csv = f"output/{nombre_base}.csv"

df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(ruta_csv)

print("✅ CSV guardado en:", ruta_csv)

✅ CSV guardado en: output/clientes_20260405_1906_seed42.csv


11.2  Exportar JSON

In [111]:
ruta_json = f"output/{nombre_base}.json"

df.coalesce(1).write \
    .mode("overwrite") \
    .json(ruta_json)

print("✅ JSON guardado en:", ruta_json)

✅ JSON guardado en: output/clientes_20260405_1906_seed42.json


# 11.1 Código de Cierre: Perfilado, Logging y Salidas Finales

In [112]:
import os
import shutil
import glob
import logging
from datetime import datetime
from pyspark.sql import functions as F

# 1. CONFIGURACIÓN DE LOGGING ESTRUCTURADO
# ---------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [TDM_ENGINE] %(message)s')
logger = logging.getLogger("FinalDelivery")

def finalizar_entrega(df, nombre_base):
    logger.info(f"INICIO_FINALIZACION: Procesando entregables para {nombre_base}")

    # 2. REPORTE DE PERFILADO (Profiling)
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("      PERFILADO ESTADÍSTICO DE SALIDA (TDM)      ")
    print("="*50)

    # Distribución de estados para verificar diversidad
    print("\n📊 Distribución de Estado Cliente:")
    df.groupBy("estado_cliente").count().show()

    # Verificación de integridad de IDs
    conteo_ids = df.groupBy("tipo_identificacion").count()
    print("🆔 Resumen de Identificaciones Generadas:")
    conteo_ids.show()

    logger.info("PERFILADO_COMPLETADO: Estadísticas de distribución generadas.")

    # 3. GESTIÓN DE ARCHIVOS DE SALIDA (CSV/JSON)
    # ---------------------------------------------------------
    try:
        # Carpeta temporal generada por Spark
        ruta_csv_tmp = f"output/{nombre_base}.csv"
        ruta_json_tmp = f"output/{nombre_base}.json"

        # Buscar y copiar archivo real CSV
        csv_generado = glob.glob(f"{ruta_csv_tmp}/part-*.csv")[0]
        csv_final = f"{nombre_base}.csv"
        shutil.copy(csv_generado, csv_final)
        logger.info(f"FILE_EXPORT: CSV final creado -> {csv_final}")

        # Buscar y copiar archivo real JSON
        json_generado = glob.glob(f"{ruta_json_tmp}/part-*.json")[0]
        json_final = f"{nombre_base}.json"
        shutil.copy(json_generado, json_final)
        logger.info(f"FILE_EXPORT: JSON final creado -> {json_final}")

        print(f"\n✅ PROCESO FINALIZADO CON ÉXITO")
        print(f"📦 Entregable CSV: {csv_final}")
        print(f"📦 Entregable JSON: {json_final}")

    except Exception as e:
        logger.error(f"ERROR_EXPORTACION: Fallo al consolidar archivos finales: {str(e)}")

# EJECUCIÓN DEL CIERRE
finalizar_entrega(df, nombre_base)


      PERFILADO ESTADÍSTICO DE SALIDA (TDM)      

📊 Distribución de Estado Cliente:
+--------------+-----+
|estado_cliente|count|
+--------------+-----+
|        ACTIVO|  215|
|      INACTIVO|  185|
+--------------+-----+

🆔 Resumen de Identificaciones Generadas:
+-------------------+-----+
|tipo_identificacion|count|
+-------------------+-----+
|                RUC|  124|
|          PASAPORTE|  145|
|             CEDULA|  131|
+-------------------+-----+


✅ PROCESO FINALIZADO CON ÉXITO
📦 Entregable CSV: clientes_20260405_1906_seed42.csv
📦 Entregable JSON: clientes_20260405_1906_seed42.json


# 12 Creación archivo Readme

In [2]:
# Definimos el contenido del README usando lenguaje Markdown
readme_content = """
# 🚀 Generador de Datos Sintéticos - Unidad de Entornos No Productivos

Este proyecto automatiza la creación de datasets de clientes bajo un contrato de datos explícito (YAML).

## 🏗️ Arquitectura de la Solución
- **Gobernanza**: Uso de `contract.yaml` para definir reglas de negocio.
- **Procesamiento**: Motor en **PySpark** para generación escalable.
- **Calidad**: Validador determinístico con reporte de métricas (Schema, Domain, Dup, Business).

## 🛠️ Instrucciones de Ejecución
1. Instalar dependencias: `pip install pyspark pyyaml`
2. Cargar el archivo `contract.yaml`.
3. Ejecutar el notebook para generar los archivos `CSV` y `JSON`.

## 🧪 Reproducibilidad
Para obtener resultados consistentes y trazables, se utiliza el **Seed 42**.
"""

# Guardamos el archivo en el entorno de Colab
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ Archivo README.md creado en el panel de archivos (izquierda).")

✅ Archivo README.md creado en el panel de archivos (izquierda).


Readme + Arquitectura + Presentación Proyecto Tdm
Nombre: Patricia Valarezo
📊 Proyecto: Validación Determinística de Datos (TDM)
🧩 Descripción del Proyecto

Este proyecto implementa un validador determinístico de calidad de datos sobre datasets generados sintéticamente, enfocado en entornos de Test Data Management (TDM).

El objetivo es garantizar que los datos cumplan reglas de calidad mediante validaciones estructuradas y generar métricas claras para análisis.

🎯 Objetivos
Validar datasets generados automáticamente
Detectar errores introducidos por error-rate
Clasificar errores por tipo:
Schema
Domain
Duplicados
Reglas de negocio
Generar métricas de calidad:
total_registros
reglas_evaluadas
errores_totales
errores_por_regla
% cumplimiento
muestras de errores
🏗️ Arquitectura
🔹 Arquitectura de Referencia (TDM)
[Data Generator] → [Raw Data]
        ↓
[Data Validation Engine]
        ↓
[Quality Metrics & Report]
        ↓
[Consumo: BI / Auditoría / Testing]
🔹 Arquitectura de Solución
PySpark
│
├── Ingesta de datos (DataFrame)
├── Reglas de validación
│     ├── Schema
│     ├── Dominio
│     ├── Duplicados
│     └── Negocio
│
├── Clasificación de errores
├── Métricas de calidad
│
└── Output
      ├── Reporte estructurado
      └── Dataset con errores
⚙️ Tecnologías
Python 3.10+
PySpark
Pandas (opcional)
Google Colab / Databricks
🚀 Instalación
pip install pyspark pandas
▶️ Ejecución
Cargar dataset:
spark.read.csv("data.csv", header=True, inferSchema=True)
Ejecutar validador
Generar reporte
🧪 Ejemplo de Validaciones
1. Schema
Campos nulos
2. Dominio
Valores fuera de rango
3. Duplicados
Registros repetidos
4. Negocio
Reglas específicas